In [ ]:

!pip install -q openai-whisper
!apt-get install -y ffmpeg


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:6.1.1-3ubuntu5).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.


In [ ]:

import whisper

print("Whisper imported successfully!")


Whisper imported successfully!


In [ ]:

model = whisper.load_model("small")

print("Whisper-small loaded successfully!")


100%|███████████████████████████████████████| 461M/461M [00:05<00:00, 81.3MiB/s]


Whisper-small loaded successfully!


In [ ]:

from google.colab import files

# Upload your audio recording
uploaded = files.upload()

# Get the uploaded audio filename
audio_path = next(iter(uploaded))

print("Audio uploaded:", audio_path)


Saving New Recording.mp3 to New Recording.mp3
Audio uploaded: New Recording.mp3


In [ ]:

# Convert speech into English text
result = model.transcribe(
    audio_path,
    language="en",
    task="transcribe"
)

# Display the transcription
print("Recognised speech:")
print(result["text"])


/usr/local/lib/python3.13/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Recognised speech:
 Hello, I'm testing a speech recognition system designed to make communication easier for deaf or hard of hearing people.


In [ ]:

!pip install -q gradio


In [ ]:

import gradio as gr

def transcribe_audio(audio_path, previous_captions):
    if audio_path is None:
        return previous_captions, "Please record audio first."

    status = "Transcribing..."

    result = model.transcribe(
        audio_path,
        language="en",
        task="transcribe"
    )

    new_text = result["text"].strip()

    if previous_captions:
        updated_captions = previous_captions + "\n\n" + new_text
    else:
        updated_captions = new_text

    return updated_captions, "Transcription complete!"

def clear_captions():
    return "", "Ready to record."

with gr.Blocks(
    theme=gr.themes.Soft(),
    css="#captions textarea {font-size: 26px !important; line-height: 1.5 !important;}"
) as demo:

    gr.Markdown("# Speech-to-Text Accessibility")
    gr.Markdown("Speak into the microphone to display your words.")

    audio_input = gr.Audio(
        sources=["microphone"],
        type="filepath",
        label="Microphone"
    )

    transcribe_button = gr.Button(
        "Transcribe speech",
        variant="primary"
    )

    status = gr.Textbox(
        label="Status",
        value="Ready to record.",
        interactive=False
    )

    captions = gr.Textbox(
        label="Conversation captions",
        lines=8,
        interactive=False,
        elem_id="captions"
    )

    transcribe_button.click(
        fn=transcribe_audio,
        inputs=[audio_input, captions],
        outputs=[captions, status]
    )

    clear_button = gr.Button("Clear captions")
    clear_button.click(
        fn=clear_captions,
        outputs=[captions, status]
    )

demo.launch(share=True)


/tmp/ipykernel_1779/2633094538.py:27: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://565e84a256268fe210.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:

def test_transcription(audio_path):
    if audio_path is None:
        return "Please upload an audio recording."

    result = model.transcribe(
        audio_path,
        language="en",
        task="transcribe"
    )

    print("Recognised language:", result["language"])
    print("Average log probability:", result["segments"][0]["avg_logprob"]
          if result["segments"] else "Unavailable")

    return result["text"]

test_demo = gr.Interface(
    fn=test_transcription,
    inputs=gr.Audio(
        sources=["microphone", "upload"],
        type="filepath",
        label="Record or upload audio"
    ),
    outputs=gr.Textbox(
        label="Transcription",
        lines=5
    ),
    title="Whisper Accuracy Testing"
)

test_demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c87ad538295ad505f3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:

import os
import tempfile
import whisper

def transcribe_chunk(audio_path):
    if audio_path is None:
        return ""

    result = model.transcribe(
        audio_path,
        language="en",
        task="transcribe",
        fp16=False
    )

    return result["text"].strip()

print("Chunk transcription function ready!")


Chunk transcription function ready!


In [ ]:

import gradio as gr
import numpy as np

def check_audio(audio):
    if audio is None:
        return "Waiting for microphone audio..."

    sample_rate, audio_data = audio

    duration = len(audio_data) / sample_rate

    return (
        f"Microphone is streaming!\n"
        f"Sample rate: {sample_rate} Hz\n"
        f"Audio received: {duration:.2f} seconds"
    )

with gr.Blocks() as test_app:

    gr.Markdown("# Microphone Streaming Test")

    microphone = gr.Audio(
        sources=["microphone"],
        type="numpy",
        streaming=True,
        label="Speak into your microphone"
    )

    status = gr.Textbox(
        label="Streaming status",
        lines=3
    )

    microphone.stream(
        fn=check_audio,
        inputs=microphone,
        outputs=status,
        stream_every=0.5,
        time_limit=60
    )

test_app.queue()
test_app.launch(share=True)


ERROR:asyncio:Task was destroyed but it is pending!
task: <Task pending name='Task-642' coro=<Queue.wait_for_event() done, defined at /usr/local/lib/python3.13/dist-packages/gradio/queueing.py:726> wait_for=<Future pending cb=[Task.task_wakeup()]>>


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8ec5ddedaee586b808.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import threading
import time

import gradio as gr
import numpy as np
from scipy.signal import resample_poly

# ---------------- Settings you can tune ----------------
WHISPER_RATE = 16000       # Whisper expects 16 kHz audio
SILENCE_LEVEL = 0.01       # mic level below this counts as silence (watch "Mic level" in the status box)
PAUSE_SECONDS = 0.8        # a pause this long finishes the current caption line
MAX_PHRASE_SECONDS = 15    # start a new line if someone talks this long without pausing
MIN_SECONDS = 1.0          # wait for at least this much audio before transcribing
USE_FP16 = model.device.type == "cuda"   # half precision only works on a GPU

if not USE_FP16:
    print("Whisper is running on CPU, so captions will lag a few seconds behind your voice.")
    print("For real-time captions: Runtime > Change runtime type > T4 GPU, then re-run all cells.")

# ---------------- Shared state ----------------
lock = threading.Lock()
live = {
    "sr": None,                          # microphone sample rate
    "audio": np.zeros(0, np.float32),    # audio of the phrase currently being spoken
    "received": 0,                       # total samples received from the mic
    "processed": 0,                      # value of "received" at the last transcription
    "lines": [],                         # finished caption lines
    "partial": "",                       # caption for the phrase still being spoken
    "took": 0.0,                         # how long the last transcription took
    "epoch": 0,                          # goes up by one every time captions are cleared
}


def captions_text():
    lines = live["lines"] + ([live["partial"]] if live["partial"] else [])
    return "\n".join(lines)


def frame_levels(audio, sr, frame_seconds=0.1):
    """Loudness (RMS) of each 0.1 s slice of audio."""
    n = int(sr * frame_seconds)
    if len(audio) < n:
        return np.zeros(1, np.float32)
    frames = audio[: len(audio) // n * n].reshape(-1, n)
    return np.sqrt((frames ** 2).mean(axis=1))


def receive_audio(chunk):
    """Runs every 0.5 s while recording. It only stores the new audio, so it is always fast."""
    if chunk is None:
        return "Waiting for microphone audio..."

    sr, data = chunk

    if np.issubdtype(data.dtype, np.integer):
        data = data.astype(np.float32) / np.iinfo(data.dtype).max
    else:
        data = data.astype(np.float32)
    if data.ndim > 1:                    # stereo -> mono
        data = data.mean(axis=1)

    with lock:
        if live["sr"] != sr:
            live["sr"] = sr
            live["audio"] = np.zeros(0, np.float32)
        live["audio"] = np.concatenate([live["audio"], data])
        live["received"] += len(data)
        heard = live["received"] / sr
        took = live["took"]

    level = float(np.sqrt(np.mean(data ** 2))) if len(data) else 0.0
    return f"Listening... {heard:.0f} s heard | Mic level: {level:.3f} | Last caption update took {took:.1f} s"


def update_captions():
    """Runs on a timer. Re-transcribes the phrase being spoken and updates the captions."""
    with lock:
        sr = live["sr"]
        audio = live["audio"].copy()
        received = live["received"]
        epoch = live["epoch"]
        nothing_new = received == live["processed"]

    if sr is None or nothing_new or len(audio) < MIN_SECONDS * sr:
        return gr.skip()

    levels = frame_levels(audio, sr)

    # Only silence so far: throw it away (keeping a short tail) instead of asking Whisper,
    # which tends to invent words when it is given silence.
    if levels.max() < SILENCE_LEVEL:
        keep = int(0.3 * sr)
        with lock:
            if live["epoch"] == epoch:
                live["audio"] = live["audio"][max(0, len(audio) - keep):]
                live["processed"] = received
        return gr.skip()

    started = time.time()
    audio_16k = audio if sr == WHISPER_RATE else resample_poly(audio, WHISPER_RATE, sr).astype(np.float32)
    result = model.transcribe(
        audio_16k,
        language="en",
        task="transcribe",
        fp16=USE_FP16,
        temperature=0.0,                     # single pass = faster, steadier captions
        condition_on_previous_text=False
    )
    text = result["text"].strip()
    segments = result["segments"]

    pause_frames = int(PAUSE_SECONDS / 0.1)
    paused = len(levels) > pause_frames and levels[-pause_frames:].max() < SILENCE_LEVEL
    too_long = len(audio) / sr > MAX_PHRASE_SECONDS

    with lock:
        if live["epoch"] != epoch:           # captions were cleared while we were transcribing
            return captions_text()

        if paused or (too_long and len(segments) < 2):
            # The speaker paused: this phrase is finished, so lock it in as a caption line.
            if text:
                live["lines"].append(text)
            live["partial"] = ""
            live["audio"] = live["audio"][len(audio):]
        elif too_long:
            # Long non-stop speech: lock in everything except the last sentence and keep going.
            finished = "".join(s["text"] for s in segments[:-1]).strip()
            if finished:
                live["lines"].append(finished)
            live["partial"] = segments[-1]["text"].strip()
            live["audio"] = live["audio"][int(segments[-1]["start"] * sr):]
        else:
            # Still mid-phrase: show the latest guess, it will be refined on the next update.
            live["partial"] = text

        live["processed"] = received
        live["took"] = time.time() - started
        return captions_text()


def clear_captions():
    with lock:
        live["audio"] = np.zeros(0, np.float32)
        live["lines"] = []
        live["partial"] = ""
        live["processed"] = live["received"]
        live["epoch"] += 1
    return "", "Captions cleared."


gr.close_all()   # shut down the earlier test apps so they don't keep running in the background

with gr.Blocks() as live_app:

    gr.Markdown("# Live Speech-to-Text Captions")
    gr.Markdown("Press record and start speaking. Captions appear while you talk.")

    microphone = gr.Audio(
        sources=["microphone"],
        type="numpy",
        streaming=True,
        label="Speak into your microphone"
    )

    status = gr.Textbox(
        label="Status",
        value="Ready. Press record to start.",
        interactive=False
    )

    captions = gr.Textbox(
        label="Live captions",
        lines=8,
        interactive=False,
        elem_id="captions"
    )

    clear_button = gr.Button("Clear captions")

    timer = gr.Timer(0.5)

    # 1) Collect microphone audio every 0.5 s
    microphone.stream(
        fn=receive_audio,
        inputs=microphone,
        outputs=status,
        stream_every=0.5,
        time_limit=600,              # recording stops after 10 minutes; press record again to continue
        show_progress="hidden"
    )

    # 2) Refresh the captions twice a second (skips a beat if Whisper is still busy)
    timer.tick(
        fn=update_captions,
        outputs=captions,
        show_progress="hidden"
    )

    clear_button.click(
        fn=clear_captions,
        outputs=[captions, status]
    )

live_app.queue()
live_app.launch(
    share=True,
    css="#captions textarea {font-size: 26px !important; line-height: 1.5 !important;}"
)


Whisper is running on CPU, so captions will lag a few seconds behind your voice.
For real-time captions: Runtime > Change runtime type > T4 GPU, then re-run all cells.
Closing server running on port: 7861
Closing server running on port: 7865
Closing server running on port: 7864
Closing server running on port: 7862
Closing server running on port: 7860
Closing server running on port: 7863
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://76392a4c760cf453cb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
